# Ensemble Model-1-2 (Individual characteristics + CPET Variables)

Use the Python 3.11 kernel and private preprocessed data; see [setup](../README.md). Run cells top to bottom.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from ai_cpet_ots.dataloaders import Model_1_2_dataset, load_clinical_data, load_CPET_data
from ai_cpet_ots.models import Model_1, Model_1_2, Model_2
from ai_cpet_ots.paths import (
    CLIN_CONTROL_DIR,
    CLIN_OTS_DIR,
    CPET_CONTROL_DIR,
    CPET_OTS_DIR,
    MODEL_WEIGHTS,
    output_path,
)
from ai_cpet_ots.testers import test_model_1_2

In [ ]:
def optimize_model(X_train_clinical, Y_train_clinical, X_train_cpet, Y_train_cpet):
    """Search the notebook's study hyperparameter bounds using training inputs only.

    Returns the optimizer result used by train_model. Requires the optimization
    extra and can be expensive; it is not run in the default evaluation workflow."""
    from ai_cpet_ots.optimizers import Optimizer_Model_1_2

    # Optimize model
    opt = Optimizer_Model_1_2(X_train_clinical, Y_train_clinical, X_train_cpet, Y_train_cpet)

    bounds = [
        {"name": "n_epochs", "type": "discrete", "domain": np.arange(5, 25, 5)},
        {"name": "dropout", "type": "discrete", "domain": np.arange(0, 1, 0.1)},
        {"name": "lr", "type": "discrete", "domain": (1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1)},
        {
            "name": "weight_decay",
            "type": "discrete",
            "domain": (0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1),
        },
        {"name": "batch_size", "type": "discrete", "domain": [16, 32, 64, 128]},
        {"name": "n_hidden_layers", "type": "discrete", "domain": np.arange(1, 6, 1)},
        {"name": "n_nodes", "type": "discrete", "domain": [16, 32, 64, 128]},
    ]

    _, opt_params_gpbo = opt.run_bo_gp(bounds, 10)

    return opt_params_gpbo

In [ ]:
def train_model(opt_params, X_train_clinical, Y_train_clinical, X_train_cpet, Y_train_cpet):
    """Train this notebook's model from opt_params and the supplied training arrays.

    Keeps the original objective, optimizer, and epoch settings. Returns the
    trained model; checkpoint saving is controlled separately by save_model."""
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else:
        device = torch.device("cpu")

    n_epochs, dropout, lr, weight_decay, batch_size, n_hidden_layers, n_nodes, threshold = (
        opt_params
    )

    n_epochs = int(n_epochs)
    n_hidden_layers = int(n_hidden_layers)
    n_nodes = int(n_nodes)
    batch_size = int(batch_size)

    # Shuffle trainset and create minibatch
    trainset = Model_1_2_dataset(X_train_clinical, Y_train_clinical, X_train_cpet, Y_train_cpet)
    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)

    # Initializing models
    model_1 = Model_1().to(device)
    if device.type == "cpu":
        model_1.load_state_dict(
            torch.load(
                MODEL_WEIGHTS / "Model_1.pth", map_location=torch.device("cpu"), weights_only=True
            )
        )
    else:
        model_1.load_state_dict(torch.load(MODEL_WEIGHTS / "Model_1.pth", weights_only=True))
    model_1.eval()

    model_2 = Model_2().to(device)
    if device.type == "cpu":
        model_2.load_state_dict(
            torch.load(
                MODEL_WEIGHTS / "Model_2.pth", map_location=torch.device("cpu"), weights_only=True
            )
        )
    else:
        model_2.load_state_dict(torch.load(MODEL_WEIGHTS / "Model_2.pth", weights_only=True))
    model_2.eval()

    model = Model_1_2(dropout, n_hidden_layers, n_nodes, threshold).to(device)

    pos_samples = np.sum(Y_train)
    neg_samples = len(Y_train) - pos_samples
    pos_weight = torch.tensor([neg_samples / pos_samples]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Training
    for e in range(n_epochs):
        train_loss = 0
        for batch in trainloader:
            x_batch_clinical, y_batch, x_batch_cpet, _ = batch
            x_batch_clinical, y_batch, x_batch_cpet = (
                x_batch_clinical.float().to(device),
                y_batch.to(device),
                x_batch_cpet.float().to(device),
            )

            with torch.no_grad():
                _, feat_clinical = model_1(x_batch_clinical)
                _, feat_cpet = model_2(x_batch_cpet)
            feature_maps = torch.concatenate((feat_clinical, feat_cpet), axis=1)

            y_pred = model(feature_maps)  # raw logits

            loss = criterion(y_pred[0].flatten(), y_batch)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

        train_loss = loss.item() / len(trainloader)

        print(f"Epoch [{e + 1}/{n_epochs}] - Loss {train_loss:.4f}")

    return model

In [ ]:
def test_model(
    X_test_clinical, Y_test_clinical, X_test_cpet, Y_test_cpet, model=None, save_fig=False
):
    """Load the published checkpoint when model is None and evaluate test inputs.

    Constructs the study DataLoader and forwards plot options to the reusable
    evaluator. Private inputs must use the original feature order and scaling."""
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else:
        device = torch.device("cpu")

    model_1 = Model_1().to(device)
    model_2 = Model_2().to(device)

    model_1.load_state_dict(
        torch.load(MODEL_WEIGHTS / "Model_1.pth", map_location=device, weights_only=True)
    )
    model_2.load_state_dict(
        torch.load(MODEL_WEIGHTS / "Model_2.pth", map_location=device, weights_only=True)
    )

    model_1.eval()
    model_2.eval()

    if model is None:
        model = Model_1_2().to(device)
        model.load_state_dict(
            torch.load(MODEL_WEIGHTS / "Model_1_2.pth", map_location=device, weights_only=True)
        )

    testset = Model_1_2_dataset(X_test_clinical, Y_test_clinical, X_test_cpet, Y_test_cpet)
    testloader = DataLoader(testset, batch_size=len(testset), shuffle=False)

    if save_fig:
        print("Saving fig...")

    auroc, auprc, f1, acc = test_model_1_2(
        model=model,
        model_1=model_1,
        model_2=model_2,
        model_name="Model 1-2",
        loader=testloader,
        use_pred=False,
        plot_cm=True,
        plot_PRC=True,
        plot_predictions=True,
        plot_ROC=True,
        save_fig=save_fig,
    )
    print(
        f"AUROC: {auroc:.3f}, AUCPR: {auprc:.3f}, F1: {f1:.3f}, ACC: {acc:.3f}, Threshold {model.threshold:.3f}"
    )

In [ ]:
training_model = False
save_model = False
save_fig = True
model_name = "Model_1_2"

In [ ]:
_, _, X_train_clinical, Y_train_clinical, X_test_clinical, Y_test_clinical = load_clinical_data(
    CLIN_CONTROL_DIR, CLIN_OTS_DIR
)

In [ ]:
_, _, X_train_cpet, Y_train_cpet, X_test_cpet, Y_test_cpet = load_CPET_data(
    CPET_CONTROL_DIR, CPET_OTS_DIR
)

In [ ]:
X_train = np.concatenate((X_train_clinical, X_train_cpet), axis=1)
Y_train = Y_train_clinical
X_test = np.concatenate((X_test_clinical, X_test_cpet), axis=1)
Y_test = Y_test_clinical
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

In [ ]:
if training_model:
    # Optimize model
    opt_params = optimize_model(X_train_clinical, Y_train_clinical, X_train_cpet, Y_train_cpet)

    # Train the model
    model = train_model(opt_params, X_train_clinical, Y_train_clinical, X_train_cpet, Y_train_cpet)

    # Test model
    test_model(X_test_clinical, Y_test_clinical, X_test_cpet, Y_test_cpet, model)

    if save_model:
        torch.save(model.state_dict(), output_path(f"{model_name}.pth"))
else:
    test_model(X_test_clinical, Y_test_clinical, X_test_cpet, Y_test_cpet, save_fig=save_fig)